
# Scheduler Instrumentation

## Objective

The goal is to make the vLLM scheduler observable.

The source-level execution path has already been traced as:

```text
Request
    ↓
Scheduler.schedule()
    ↓
token-budget decision
    ↓
KV-cache allocation
    ↓
ModelRunner
```

The next step is to add lightweight instrumentation around this path so that scheduler decisions can be inspected at runtime.

The trace should answer:

```text
Which request was scheduled?

How many tokens did it receive?

Was the request doing prefill or decode?

How much scheduler budget remained?

How much KV-cache capacity remained?

How many requests were waiting and running?

Did preemption occur?
```

---

## Trace Format

Scheduler events will be written in JSON Lines format:

```text
scheduler_trace.jsonl
```

Each line represents one scheduler event.

Example:

```json
{
  "step": 12,
  "event": "request_scheduled",
  "request_id": "3",
  "num_tokens": 3000,
  "num_computed_tokens": 1024,
  "num_new_tokens": 1024,
  "is_prefill": true,
  "token_budget_before": 2048,
  "token_budget_after": 1024,
  "free_kv_blocks": 8241,
  "waiting_size": 2,
  "running_size": 3,
  "num_preemptions": 0
}
```

---

## Core Fields

### Scheduler-Level Fields

```text
step
event
token_budget_before
token_budget_after
free_kv_blocks
waiting_size
running_size
```

#### step

Scheduler iteration number.

This provides a simple ordering for scheduler decisions across repeated calls to:

```python
Scheduler.schedule()
```

#### event

Type of scheduler event.

Initial event types:

```text
schedule_start
request_scheduled
request_preempted
schedule_end
```

Additional events can be added later if required.

#### token_budget_before

Available scheduler token budget before scheduling a request.

#### token_budget_after

Remaining scheduler token budget after scheduling a request.

#### free_kv_blocks

Number of currently free physical KV-cache blocks.

This value can be obtained through the KV-cache manager or underlying block pool.

#### waiting_size

Number of requests currently waiting for scheduling.

#### running_size

Number of active running requests.

---

### Request-Level Fields

```text
request_id
status
num_tokens
num_computed_tokens
num_new_tokens
is_prefill
is_prefill_chunk
num_preemptions
```

#### request_id

Unique request identifier.

#### status

Current request scheduler status.

Examples:

```text
WAITING
RUNNING
PREEMPTED
FINISHED
```

#### num_tokens

Current logical sequence length:

```text
prompt tokens
+
generated output tokens
```

#### num_computed_tokens

Number of tokens that the scheduler currently considers computed.

#### num_new_tokens

Number of tokens assigned to the request in the current scheduler iteration.

#### is_prefill

Simple workload classification.

Conceptually:

```text
num_computed_tokens < num_prompt_tokens
→ prefill

otherwise
→ decode
```

This field is mainly included to make traces easier to read.

#### is_prefill_chunk

Uses the scheduler's existing request state:

```python
request.is_prefill_chunk
```

This indicates whether the request still has unfinished prefill work after the current scheduling decision.

#### num_preemptions

Number of times the request has been preempted.

---

## Initial Event Types

### 1. `schedule_start`

Written once at the beginning of each scheduler iteration.

Example:

```json
{
  "step": 5,
  "event": "schedule_start",
  "token_budget_before": 2048,
  "free_kv_blocks": 12034,
  "waiting_size": 3,
  "running_size": 2
}
```

---

### 2. `request_scheduled`

Written whenever a request successfully receives tokens.

Example:

```json
{
  "step": 5,
  "event": "request_scheduled",
  "request_id": "req-2",
  "num_computed_tokens": 1024,
  "num_new_tokens": 512,
  "is_prefill": true,
  "token_budget_before": 2048,
  "token_budget_after": 1536,
  "free_kv_blocks": 11980
}
```

---

### 3. `request_preempted`

Written when KV-cache pressure causes a running request to be preempted.

Example:

```json
{
  "step": 17,
  "event": "request_preempted",
  "request_id": "req-7",
  "num_computed_tokens": 3072,
  "num_preemptions": 1,
  "free_kv_blocks": 4
}
```

---

### 4. `schedule_end`

Written once after the scheduler finishes constructing the current execution plan.

Example:

```json
{
  "step": 5,
  "event": "schedule_end",
  "scheduled_requests": 3,
  "scheduled_tokens": 1825,
  "token_budget_after": 223,
  "free_kv_blocks": 11890,
  "waiting_size": 1,
  "running_size": 4
}
```

---

## Initial Instrumentation Scope

The first version of the instrumentation should remain intentionally small.

We will initially trace only:

```text
scheduler step
request ID
scheduled token count
prefill/decode state
token budget
KV free blocks
waiting/running queue size
preemption
```

We will not initially trace:

```text
per-layer KV usage
attention backend metrics
GPU kernel timing
CUDA events
per-request memory bytes
speculative decoding acceptance rate
```

These can be added later if they become useful for the performance analysis.

---

## Expected Runtime View

The trace should allow scheduler behavior to be reconstructed in a readable form such as:

```text
Step 1
A: prefill 1024
remaining budget: 1024

Step 2
A: prefill 1024
B: prefill 128
remaining budget: 896

Step 3
A: decode 1
B: decode 1
C: prefill 512
```

This converts the scheduler from a source-code abstraction into observable runtime behavior.

---

## Success Criteria

The instrumentation is considered complete when:

1. Scheduler events are written to JSONL.
2. A controlled mixed workload produces a valid trace.
3. The trace shows both prefill and decode scheduling.
4. Token-budget transitions are internally consistent.
5. KV free-block counts are visible.
6. Preemption events can be recorded when they occur.
7. Request outputs remain correct with tracing enabled.


## 0. Confirm the GPU

In [1]:
!nvidia-smi

Sun Sep 20 15:43:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.6 MB/s eta 0:00:00


In [3]:
# vLLM changes quickly. For Day 1 we use the released package and record
# the exact installed version below so future benchmarks are reproducible.
%pip install -q -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 437.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
import os
os.kill(os.getpid(), 9)

## 1. schedule_start

### Inspect Scheduler Instrumentation Points

In [1]:
# =============================================================================
# Inspect Scheduler.schedule()
# =============================================================================

import inspect
import vllm.v1.core.sched.scheduler as scheduler_module

Scheduler = scheduler_module.Scheduler

print(inspect.getfile(Scheduler))
print("\n" + "=" * 100 + "\n")

source = inspect.getsource(Scheduler.schedule)
print(source)

/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py


    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
        self.current_step += 1
        # NOTE(woosuk) on the scheduling algorithm:
        # There's no "decoding phase" nor "prefill phase" in the scheduler.
        # Each request just has the num_computed_tokens and
        # num_tokens_with_spec. num_tokens_with_spec =
        # len(prompt_token_ids) + len(output_token_ids) + len(spec_token_ids).
        # At each step, the scheduler tries to assign tokens to the requests
        # so that each request's num_computed_tokens can catch up its
        # num_tokens_with_spec. This is general enough to cover
        # chunked prefills, prefix caching, speculative decoding,
        # and the "jump decoding" optimization in the future.

        scheduled_new_reqs: list[Request] = []
        scheduled_resumed_reqs: list[Request] = []
        scheduled_running_reqs: list[Request] = []
        

In [2]:
# =============================================================================
# Locate important scheduler instrumentation points
# =============================================================================

source = inspect.getsource(Scheduler.schedule)
lines = source.splitlines()

keywords = [
    "token_budget",
    "num_new_tokens",
    "num_scheduled_tokens",
    "allocate_slots",
    "_preempt_request",
    "preempted_req",
    "self.running.append",
    "request.status",
    "SchedulerOutput",
]

for i, line in enumerate(lines):
    if any(keyword in line for keyword in keywords):
        print(f"{i:4d}: {line}")

   0:     def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
  16:         preempted_reqs: list[Request] = []
  19:         num_scheduled_tokens: dict[str, int] = {}
  20:         token_budget = self.max_num_scheduled_tokens
  26:             token_budget = 0
  51:         while req_index < len(self.running) and token_budget > 0:
  84:             num_new_tokens = (
  89:             if 0 < self.scheduler_config.long_prefill_token_threshold < num_new_tokens:
  90:                 num_new_tokens = self.scheduler_config.long_prefill_token_threshold
  91:             num_new_tokens = min(
  92:                 num_new_tokens, token_budget, input_budget - draft_slots
  97:             num_new_tokens = min(
  98:                 num_new_tokens,
 106:                 num_new_tokens = self._mamba_block_aligned_split(
 107:                     request, num_new_tokens
 117:                     num_new_tokens,
 123:                     num_new_tokens,
 130:             num_n

In [3]:
# =============================================================================
# Inspect KV-cache free-block APIs
# =============================================================================

import inspect
import vllm.v1.core.kv_cache_manager as kv_module
import vllm.v1.core.block_pool as block_pool_module

print("KVCacheManager methods:")
for name, obj in inspect.getmembers(kv_module.KVCacheManager):
    if "free" in name.lower() or "block" in name.lower():
        if callable(obj):
            print(name)

print("\nBlockPool methods:")
for name, obj in inspect.getmembers(block_pool_module.BlockPool):
    if "free" in name.lower() or "block" in name.lower():
        if callable(obj):
            print(name)

KVCacheManager methods:
cache_blocks
create_kv_cache_blocks
evict_blocks
free
get_block_ids
get_block_ids_for_computed_tokens
get_blocks
get_computed_blocks
get_computed_blocks_for_connector
get_num_common_prefix_blocks
get_zeroing_block_ids_in_range
pop_blocks_for_free
record_blocks_for_zeroing
remove_skipped_blocks
take_kv_cache_block_copies
take_new_block_ids
truncate_computed_blocks

BlockPool methods:
_build_block_stored_event
_emit_block_removed_events
_get_partial_block_hash
_get_partial_block_parent_hash_and_start
_insert_block_hash
_maybe_evict_cached_block
_remove_cached_block_hashes
cache_full_blocks
cache_partial_block
emit_cached_block_events
evict_blocks
free_blocks
get_cached_block
get_new_blocks
get_num_free_blocks
is_block_writable
move_block_hashes


## 2. request_scheduled

In [4]:
# =============================================================================
# Inspect request_scheduled insertion points
# =============================================================================

import inspect
import vllm.v1.core.sched.scheduler as scheduler_module

source = inspect.getsource(scheduler_module.Scheduler.schedule)
lines = source.splitlines()

for i, line in enumerate(lines):
    if "num_scheduled_tokens[request_id] = num_new_tokens" in line:
        print("\n" + "=" * 100)
        print(f"Insertion point around line {i}")
        print("=" * 100)

        start = max(0, i - 12)
        end = min(len(lines), i + 15)

        for j in range(start, end):
            marker = ">>>" if j == i else "   "
            print(f"{marker} {j:4d}: {lines[j]}")


Insertion point around line 225
     213:                         # No more request to preempt. Cannot schedule this request.
     214:                         break
     215: 
     216:             if new_blocks is None:
     217:                 # Cannot schedule this request.
     218:                 break
     219: 
     220:             # Schedule the request.
     221:             scheduled_running_reqs.append(request)
     222:             prefill_scheduled |= request.is_prefill_chunk
     223:             request_id = request.request_id
     224:             req_to_new_blocks[request_id] = new_blocks
>>>  225:             num_scheduled_tokens[request_id] = num_new_tokens
     226:             token_budget -= num_new_tokens
     227:             input_budget -= num_new_tokens + draft_slots
     228:             req_index += 1
     229: 
     230:             # Speculative decode related.
     231:             if request.spec_token_ids:
     232:                 num_scheduled_s

### Trace Helper Setup

In [5]:
# =============================================================================
# Inspect Scheduler initialization
# =============================================================================

import inspect
import vllm.v1.core.sched.scheduler as scheduler_module

Scheduler = scheduler_module.Scheduler

source = inspect.getsource(Scheduler.__init__)
print(source)

    def __init__(
        self,
        vllm_config: VllmConfig,
        kv_cache_config: KVCacheConfig,
        structured_output_manager: StructuredOutputManager,
        block_size: int,
        hash_block_size: int | None = None,
        mm_registry: MultiModalRegistry = MULTIMODAL_REGISTRY,
        include_finished_set: bool = False,
        log_stats: bool = False,
    ) -> None:
        self.vllm_config = vllm_config
        self.scheduler_config = vllm_config.scheduler_config
        self.cache_config = vllm_config.cache_config
        self.lora_config = vllm_config.lora_config
        self.model_uses_mrope = vllm_config.model_config.uses_mrope
        self.model_uses_xdrope = vllm_config.model_config.uses_xdrope
        self.kv_cache_config = kv_cache_config
        self.kv_events_config = vllm_config.kv_events_config
        self.parallel_config = vllm_config.parallel_config
        self.log_stats = log_stats
        self.observability_config = vllm_config.observability_confi

In [6]:
# =============================================================================
# Inspect scheduler.py imports
# =============================================================================

source_file = inspect.getfile(Scheduler)

with open(source_file, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[:120]):
    print(f"{i:4d}: {line.rstrip()}")

   0: # SPDX-License-Identifier: Apache-2.0
   1: # SPDX-FileCopyrightText: Copyright contributors to the vLLM project
   2: import itertools
   3: import time
   4: from collections import defaultdict, deque
   5: from collections.abc import Iterable
   6: from dataclasses import replace
   7: from typing import Any
   8: 
   9: from vllm.compilation.cuda_graph import CUDAGraphStat
  10: from vllm.config import KVEventsConfig, VllmConfig
  11: from vllm.distributed.ec_transfer.ec_connector.base import (
  12:     ECConnectorBase,
  13:     ECConnectorMetadata,
  14:     ECConnectorRole,
  15: )
  16: from vllm.distributed.ec_transfer.ec_connector.factory import ECConnectorFactory
  17: from vllm.distributed.kv_events import EventPublisherFactory, KVEventBatch
  18: from vllm.distributed.kv_transfer.kv_connector.factory import KVConnectorFactory
  19: from vllm.distributed.kv_transfer.kv_connector.v1 import (
  20:     KVConnectorBase_V1,
  21:     KVConnectorRole,
  22:     SupportsHM

In [7]:
# =============================================================================
# Trace helper design
# =============================================================================
import json
import os

def _trace_event(self, event: str, **fields) -> None:
    if not self._trace_enabled:
        return

    record = {
        "step": self.current_step,
        "event": event,
        **fields,
    }

    with open(self._trace_path, "a") as f:
        f.write(json.dumps(record) + "\n")

In [9]:
# =============================================================================
# Test trace helper locally
# =============================================================================

TRACE_ENABLED = True
TRACE_PATH = "/content/scheduler_trace_test.jsonl"

class DummyScheduler:
    def __init__(self):
        self._trace_enabled = TRACE_ENABLED
        self._trace_path = TRACE_PATH
        self.current_step = 1

dummy = DummyScheduler()

_trace_event(
    dummy,
    "test_event",
    request_id="req-0",
    num_new_tokens=128,
    token_budget_before=2048,
    token_budget_after=1920,
)

print(f"Trace written to: {TRACE_PATH}")

Trace written to: /content/scheduler_trace_test.jsonl


In [10]:
# =============================================================================
# Inspect generated trace
# =============================================================================

with open(TRACE_PATH, "r") as f:
    print(f.read())

{"step": 1, "event": "test_event", "request_id": "req-0", "num_new_tokens": 128, "token_budget_before": 2048, "token_budget_after": 1920}



### Integrate Trace Helper into Scheduler

In [11]:
# =============================================================================
# Patch Scheduler with trace helper infrastructure
# =============================================================================

import inspect
from pathlib import Path
import vllm.v1.core.sched.scheduler as scheduler_module

scheduler_path = Path(inspect.getfile(scheduler_module.Scheduler))

print("Scheduler file:")
print(scheduler_path)

Scheduler file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py


In [12]:
# =============================================================================
# Backup scheduler.py
# =============================================================================

backup_path = scheduler_path.with_suffix(".py.backup")

if not backup_path.exists():
    backup_path.write_text(scheduler_path.read_text())
    print(f"Backup created: {backup_path}")
else:
    print(f"Backup already exists: {backup_path}")

Backup created: /usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py.backup


In [13]:
# =============================================================================
# Add tracing infrastructure to Scheduler
# =============================================================================

source = scheduler_path.read_text()

# -------------------------------------------------------------------------
# 1. Add imports
# -------------------------------------------------------------------------
if "import json\n" not in source:
    source = source.replace(
        "import itertools\n",
        "import itertools\nimport json\nimport os\n",
        1,
    )

# -------------------------------------------------------------------------
# 2. Add trace configuration in Scheduler.__init__()
# -------------------------------------------------------------------------
init_anchor = """        self.current_step = 0
"""

trace_config = """        self.current_step = 0

        # Custom scheduler tracing.
        self._trace_enabled = (
            os.environ.get("VLLM_SCHEDULER_TRACE", "0") == "1"
        )
        self._trace_path = os.environ.get(
            "VLLM_SCHEDULER_TRACE_PATH",
            "/tmp/vllm_scheduler_trace.jsonl",
        )
"""

if "self._trace_enabled" not in source:
    if init_anchor not in source:
        raise RuntimeError("Could not find Scheduler current_step anchor.")

    source = source.replace(
        init_anchor,
        trace_config,
        1,
    )

# -------------------------------------------------------------------------
# 3. Add _trace_event() method before schedule()
# -------------------------------------------------------------------------
schedule_anchor = """    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
"""

trace_method = """    def _trace_event(self, event: str, **fields) -> None:
        if not self._trace_enabled:
            return

        record = {
            "step": self.current_step,
            "event": event,
            **fields,
        }

        with open(self._trace_path, "a") as f:
            f.write(json.dumps(record) + "\\n")

"""

if "def _trace_event(" not in source:
    if schedule_anchor not in source:
        raise RuntimeError("Could not find Scheduler.schedule() anchor.")

    source = source.replace(
        schedule_anchor,
        trace_method + schedule_anchor,
        1,
    )

scheduler_path.write_text(source)

print("Tracing infrastructure patched successfully.")

Tracing infrastructure patched successfully.


In [14]:
# =============================================================================
# Verify patched Scheduler source
# =============================================================================

patched_source = scheduler_path.read_text()

checks = [
    "import json",
    "import os",
    "self._trace_enabled",
    "self._trace_path",
    "def _trace_event(",
]

for item in checks:
    print(f"{item}: {item in patched_source}")

import json: True
import os: True
self._trace_enabled: True
self._trace_path: True
def _trace_event(: True


In [15]:
# =============================================================================
# Verify scheduler.py syntax
# =============================================================================

import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


### schedule_start

In [17]:
# =============================================================================
# Add schedule_start trace event
# =============================================================================

source = scheduler_path.read_text()

anchor = """        if self._pause_state == PauseState.PAUSED_ALL:
            # Do not schedule any requests when paused.
            token_budget = 0
"""

replacement = """        if self._pause_state == PauseState.PAUSED_ALL:
            # Do not schedule any requests when paused.
            token_budget = 0

        self._trace_event(
            "schedule_start",
            token_budget_before=token_budget,
            input_budget_before=input_budget,
            free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
            waiting_size=len(self.waiting),
            running_size=len(self.running),
        )
"""

if '"schedule_start"' not in source:
    if anchor not in source:
        raise RuntimeError("Could not find schedule_start insertion point.")

    source = source.replace(anchor, replacement, 1)
    scheduler_path.write_text(source)
    print("schedule_start trace inserted.")
else:
    print("schedule_start trace already exists.")

schedule_start trace inserted.


In [18]:
# =============================================================================
# Verify schedule_start insertion
# =============================================================================

source = scheduler_path.read_text()

idx = source.find('"schedule_start"')

print(source[idx - 300: idx + 500])

m_new_slots_for_drafting if spec is not None else 0
        input_budget = self.scheduler_config.max_num_batched_tokens
        if self._pause_state == PauseState.PAUSED_ALL:
            # Do not schedule any requests when paused.
            token_budget = 0

        self._trace_event(
            "schedule_start",
            token_budget_before=token_budget,
            input_budget_before=input_budget,
            free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
            waiting_size=len(self.waiting),
            running_size=len(self.running),
        )

        # Encoder-related.
        scheduled_encoder_inputs: dict[str, list[int]] = {}
        encoder_compute_budget = self.max_num_encoder_input_tokens
        # Spec decode-related.
        scheduled_spec_


In [19]:
# =============================================================================
# Verify scheduler.py syntax
# =============================================================================

import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


### test

In [20]:
# =============================================================================
# Prepare scheduler trace environment
# =============================================================================

import os
from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

os.environ["VLLM_SCHEDULER_TRACE"] = "1"
os.environ["VLLM_SCHEDULER_TRACE_PATH"] = TRACE_PATH

trace_file = Path(TRACE_PATH)
if trace_file.exists():
    trace_file.unlink()

print("Trace enabled:", os.environ["VLLM_SCHEDULER_TRACE"])
print("Trace path:", TRACE_PATH)

Trace enabled: 1
Trace path: /content/scheduler_trace.jsonl


In [21]:
%%writefile /content/test_scheduler_trace.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=16,
)

prompts = [
    "Explain what a GPU warp is in one sentence.",
]

outputs = llm.generate(
    prompts,
    sampling_params,
)

for output in outputs:
    print("Request ID:", output.request_id)
    print("Output:", output.outputs[0].text)

Writing /content/test_scheduler_trace.py


In [22]:
%%writefile /content/test_scheduler_trace.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=16,
)

prompts = [
    "Explain what a GPU warp is in one sentence.",
]

outputs = llm.generate(
    prompts,
    sampling_params,
)

for output in outputs:
    print("Request ID:", output.request_id)
    print("Output:", output.outputs[0].text)

Overwriting /content/test_scheduler_trace.py


In [23]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_scheduler_trace.py

INFO 09-20 16:07:13 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:07:13 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:07:13 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
config.json: 100% 660/660 [00:00<00:00, 3.48MB/s]
INFO 09-20 16:07:32 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:07:32 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:07:32 [model.py:2021] Using max model len 4096
INFO 09-20 16:07:32 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:07:33 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 19.7MB/s]


In [24]:
# =============================================================================
# Inspect scheduler trace
# =============================================================================

from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

print("Trace exists:", trace_file.exists())

if trace_file.exists():
    lines = trace_file.read_text().splitlines()

    print("Number of trace events:", len(lines))
    print()

    for line in lines[:20]:
        print(line)

Trace exists: True
Number of trace events: 18

{"step": 1, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13367, "waiting_size": 1, "running_size": 0}
{"step": 2, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13366, "waiting_size": 0, "running_size": 1}
{"step": 3, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13366, "waiting_size": 0, "running_size": 1}
{"step": 4, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13366, "waiting_size": 0, "running_size": 1}
{"step": 5, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13366, "waiting_size": 0, "running_size": 1}
{"step": 6, "event": "schedule_start", "token_budget_before": 8192, "input_budget_before": 8192, "free_kv_blocks": 13366, "waiting_size": 0, "running_siz

### Patch RUNNING request

from
"""
num_scheduled_tokens[request_id] = num_new_tokens
token_budget -= num_new_tokens
input_budget -= num_new_tokens + draft_slots
req_index += 1
"""

to
"""
num_scheduled_tokens[request_id] = num_new_tokens
token_budget -= num_new_tokens
input_budget -= num_new_tokens + draft_slots

self._trace_event(
    "request_scheduled",
    request_id=request_id,
    source_state="RUNNING",
    num_prompt_tokens=request.num_prompt_tokens,
    num_tokens=request.num_tokens,
    num_computed_tokens_before=request.num_computed_tokens,
    num_new_tokens=num_new_tokens,
    is_prefill=request.num_computed_tokens < request.num_prompt_tokens,
    is_prefill_chunk=request.is_prefill_chunk,
    token_budget_before=token_budget + num_new_tokens,
    token_budget_after=token_budget,
    free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
    waiting_size=len(self.waiting),
    running_size=len(self.running),
)

req_index += 1
"""

In [25]:
# =============================================================================
# Add request_scheduled trace for RUNNING requests
# =============================================================================

source = scheduler_path.read_text()

anchor = """            num_scheduled_tokens[request_id] = num_new_tokens
            token_budget -= num_new_tokens
            input_budget -= num_new_tokens + draft_slots
            req_index += 1
"""

replacement = """            num_scheduled_tokens[request_id] = num_new_tokens
            token_budget -= num_new_tokens
            input_budget -= num_new_tokens + draft_slots

            self._trace_event(
                "request_scheduled",
                request_id=request_id,
                source_state="RUNNING",
                num_prompt_tokens=request.num_prompt_tokens,
                num_tokens=request.num_tokens,
                num_computed_tokens_before=request.num_computed_tokens,
                num_new_tokens=num_new_tokens,
                is_prefill=request.num_computed_tokens < request.num_prompt_tokens,
                is_prefill_chunk=request.is_prefill_chunk,
                token_budget_before=token_budget + num_new_tokens,
                token_budget_after=token_budget,
                free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
                waiting_size=len(self.waiting),
                running_size=len(self.running),
            )

            req_index += 1
"""

if 'source_state="RUNNING"' not in source:
    if anchor not in source:
        raise RuntimeError("Could not find RUNNING request scheduling insertion point.")

    source = source.replace(anchor, replacement, 1)
    scheduler_path.write_text(source)
    print("RUNNING request_scheduled trace inserted.")
else:
    print("RUNNING request_scheduled trace already exists.")

RUNNING request_scheduled trace inserted.


In [26]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [27]:
# =============================================================================
# Clear old scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Old trace removed.")

Old trace removed.


In [28]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_scheduler_trace.py

INFO 09-20 16:14:38 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:14:38 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:14:38 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:14:39 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:14:39 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:14:39 [model.py:2021] Using max model len 4096
INFO 09-20 16:14:39 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:14:40 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=9541) INFO 09-20 16:14:45 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model=

In [29]:
# =============================================================================
# Inspect scheduler trace
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

lines = trace_file.read_text().splitlines()

print("Number of trace events:", len(lines))
print()

for line in lines[:50]:
    record = json.loads(line)
    print(record)

Number of trace events: 33

{'step': 1, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14341, 'waiting_size': 1, 'running_size': 0}
{'step': 2, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 2, 'event': 'request_scheduled', 'request_id': '0-95a676c1', 'source_state': 'RUNNING', 'num_prompt_tokens': 11, 'num_tokens': 11, 'num_computed_tokens_before': 11, 'num_new_tokens': 1, 'is_prefill': False, 'is_prefill_chunk': False, 'token_budget_before': 8192, 'token_budget_after': 8191, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 3, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 3, 'event': 'request_scheduled', 'request_id': '0-95a676c1', 'source_state': 'RUNNING', 'num_prompt_tokens': 11, 'num_to

### WAITING / PREEMPTED → RUNNING

In [30]:
# =============================================================================
# Add request_scheduled trace for WAITING / PREEMPTED requests
# =============================================================================

source = scheduler_path.read_text()

anchor = """                num_scheduled_tokens[request_id] = num_new_tokens
                token_budget -= num_new_tokens
                input_budget -= num_new_tokens + draft_slots
                request.status = RequestStatus.RUNNING
                request.num_computed_tokens = num_computed_tokens
"""

replacement = """                source_state = request.status.name

                num_scheduled_tokens[request_id] = num_new_tokens
                token_budget -= num_new_tokens
                input_budget -= num_new_tokens + draft_slots

                self._trace_event(
                    "request_scheduled",
                    request_id=request_id,
                    source_state=source_state,
                    num_prompt_tokens=request.num_prompt_tokens,
                    num_tokens=request.num_tokens,
                    num_computed_tokens_before=num_computed_tokens,
                    num_new_tokens=num_new_tokens,
                    is_prefill=num_computed_tokens < request.num_prompt_tokens,
                    is_prefill_chunk=(
                        num_computed_tokens + num_new_tokens
                        < request.num_prompt_tokens
                    ),
                    token_budget_before=token_budget + num_new_tokens,
                    token_budget_after=token_budget,
                    free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
                    waiting_size=len(self.waiting),
                    running_size=len(self.running),
                )

                request.status = RequestStatus.RUNNING
                request.num_computed_tokens = num_computed_tokens
"""

if 'source_state=source_state' not in source:
    if anchor not in source:
        raise RuntimeError(
            "Could not find WAITING/PREEMPTED scheduling insertion point."
        )

    source = source.replace(anchor, replacement, 1)
    scheduler_path.write_text(source)
    print("WAITING/PREEMPTED request_scheduled trace inserted.")
else:
    print("WAITING/PREEMPTED request_scheduled trace already exists.")

WAITING/PREEMPTED request_scheduled trace inserted.


In [31]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [32]:
# =============================================================================
# Clear old scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Old trace removed.")

Old trace removed.


In [33]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_scheduler_trace.py

INFO 09-20 16:17:38 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:17:38 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:17:38 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:17:39 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:17:39 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:17:39 [model.py:2021] Using max model len 4096
INFO 09-20 16:17:39 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:17:40 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=10430) INFO 09-20 16:17:45 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model

In [35]:
# =============================================================================
# Inspect scheduler trace
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

lines = trace_file.read_text().splitlines()

print("Number of trace events:", len(lines))
print()

for line in lines[:50]:
    record = json.loads(line)
    print(record)

Number of trace events: 34

{'step': 1, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14341, 'waiting_size': 1, 'running_size': 0}
{'step': 1, 'event': 'request_scheduled', 'request_id': '0-942c848e', 'source_state': 'WAITING', 'num_prompt_tokens': 11, 'num_tokens': 11, 'num_computed_tokens_before': 0, 'num_new_tokens': 11, 'is_prefill': True, 'is_prefill_chunk': False, 'token_budget_before': 8192, 'token_budget_after': 8181, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 2, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 2, 'event': 'request_scheduled', 'request_id': '0-942c848e', 'source_state': 'RUNNING', 'num_prompt_tokens': 11, 'num_tokens': 11, 'num_computed_tokens_before': 11, 'num_new_tokens': 1, 'is_prefill': False, 'is_prefill_chunk': False, 'token_budget_before': 8192, 'token_budget_afte

### schedule_end

In [36]:
# =============================================================================
# Add schedule_end trace event
# =============================================================================

source = scheduler_path.read_text()

anchor = """        assert token_budget >= 0
        assert input_budget >= 0
        assert len(self.running) <= self.max_num_running_reqs
"""

replacement = """        assert token_budget >= 0
        assert input_budget >= 0
        assert len(self.running) <= self.max_num_running_reqs

        self._trace_event(
            "schedule_end",
            total_num_scheduled_tokens=total_num_scheduled_tokens,
            scheduled_request_count=len(num_scheduled_tokens),
            token_budget_after=token_budget,
            input_budget_after=input_budget,
            free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
            waiting_size=len(self.waiting),
            running_size=len(self.running),
            preempted_count=len(preempted_reqs),
        )
"""

if '"schedule_end"' not in source:
    if anchor not in source:
        raise RuntimeError("Could not find schedule_end insertion point.")

    source = source.replace(anchor, replacement, 1)
    scheduler_path.write_text(source)

    print("schedule_end trace inserted.")
else:
    print("schedule_end trace already exists.")

schedule_end trace inserted.


In [37]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [38]:
# =============================================================================
# Clear old scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Old trace removed.")

Old trace removed.


In [39]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_scheduler_trace.py

INFO 09-20 16:23:04 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:23:04 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:23:04 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:23:05 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:23:05 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:23:05 [model.py:2021] Using max model len 4096
INFO 09-20 16:23:05 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:23:06 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=11916) INFO 09-20 16:23:12 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model

In [40]:
# =============================================================================
# Inspect scheduler trace
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

lines = trace_file.read_text().splitlines()

print("Number of trace events:", len(lines))
print()

for line in lines[:50]:
    record = json.loads(line)
    print(record)

Number of trace events: 52

{'step': 1, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14341, 'waiting_size': 1, 'running_size': 0}
{'step': 1, 'event': 'request_scheduled', 'request_id': '0-b7f11db8', 'source_state': 'WAITING', 'num_prompt_tokens': 11, 'num_tokens': 11, 'num_computed_tokens_before': 0, 'num_new_tokens': 11, 'is_prefill': True, 'is_prefill_chunk': False, 'token_budget_before': 8192, 'token_budget_after': 8181, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 1, 'event': 'schedule_end', 'total_num_scheduled_tokens': 11, 'scheduled_request_count': 1, 'token_budget_after': 8181, 'input_budget_after': 8181, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1, 'preempted_count': 0}
{'step': 2, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 2, 'event': 'request_scheduled', 'reque

### request_preempted

In [41]:
# =============================================================================
# Add request_preempted trace event
# =============================================================================

source = scheduler_path.read_text()

anchor = """                    self._preempt_request(
                        preempted_req,
                        scheduled_timestamp,
                        drop_stale_output=self.requires_kv_delivery,
                    )
                    preempted_reqs.append(preempted_req)
"""

replacement = """                    self._preempt_request(
                        preempted_req,
                        scheduled_timestamp,
                        drop_stale_output=self.requires_kv_delivery,
                    )
                    preempted_reqs.append(preempted_req)

                    self._trace_event(
                        "request_preempted",
                        request_id=preempted_req.request_id,
                        num_prompt_tokens=preempted_req.num_prompt_tokens,
                        num_tokens=preempted_req.num_tokens,
                        num_computed_tokens=preempted_req.num_computed_tokens,
                        num_preemptions=preempted_req.num_preemptions,
                        free_kv_blocks=self.kv_cache_manager.block_pool.get_num_free_blocks(),
                        waiting_size=len(self.waiting),
                        running_size=len(self.running),
                    )
"""

if '"request_preempted"' not in source:
    if anchor not in source:
        raise RuntimeError("Could not find request_preempted insertion point.")

    source = source.replace(anchor, replacement, 1)
    scheduler_path.write_text(source)
    print("request_preempted trace inserted.")
else:
    print("request_preempted trace already exists.")

request_preempted trace inserted.


In [42]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [43]:
# =============================================================================
# Clear old scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Old trace removed.")

Old trace removed.


In [44]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_scheduler_trace.py

INFO 09-20 16:26:05 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:26:05 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:26:05 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:26:06 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:26:06 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:26:06 [model.py:2021] Using max model len 4096
INFO 09-20 16:26:06 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:26:07 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=12802) INFO 09-20 16:26:12 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model

In [45]:
# =============================================================================
# Inspect scheduler trace
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

lines = trace_file.read_text().splitlines()

print("Number of trace events:", len(lines))
print()

for line in lines[:50]:
    record = json.loads(line)
    print(record)

Number of trace events: 52

{'step': 1, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14341, 'waiting_size': 1, 'running_size': 0}
{'step': 1, 'event': 'request_scheduled', 'request_id': '0-950c246d', 'source_state': 'WAITING', 'num_prompt_tokens': 11, 'num_tokens': 11, 'num_computed_tokens_before': 0, 'num_new_tokens': 11, 'is_prefill': True, 'is_prefill_chunk': False, 'token_budget_before': 8192, 'token_budget_after': 8181, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 1, 'event': 'schedule_end', 'total_num_scheduled_tokens': 11, 'scheduled_request_count': 1, 'token_budget_after': 8181, 'input_budget_after': 8181, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1, 'preempted_count': 0}
{'step': 2, 'event': 'schedule_start', 'token_budget_before': 8192, 'input_budget_before': 8192, 'free_kv_blocks': 14340, 'waiting_size': 0, 'running_size': 1}
{'step': 2, 'event': 'request_scheduled', 'reque

## 3. Mixed Workload Trace

In [46]:
%%writefile /content/test_mixed_workload.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=32,
)

long_prompt = "Explain GPU memory hierarchy in detail. " * 350

prompts = [
    long_prompt,
    "What is a GPU warp?",
    "What is KV cache?",
]

outputs = llm.generate(
    prompts,
    sampling_params,
)

for output in outputs:
    print("=" * 80)
    print("Request ID:", output.request_id)
    print("Prompt tokens:", len(output.prompt_token_ids))
    print("Output tokens:", len(output.outputs[0].token_ids))
    print("Output:", output.outputs[0].text)

Writing /content/test_mixed_workload.py


In [47]:
from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Old trace removed.")

Old trace removed.


In [48]:
!VLLM_SCHEDULER_TRACE=1 \
VLLM_SCHEDULER_TRACE_PATH=/content/scheduler_trace.jsonl \
python /content/test_mixed_workload.py

INFO 09-20 16:30:04 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:30:04 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:30:04 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:30:06 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:30:06 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:30:06 [model.py:2021] Using max model len 4096
INFO 09-20 16:30:06 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:30:07 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=13909) INFO 09-20 16:30:12 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model

In [49]:
# =============================================================================
# Inspect mixed-workload scheduling decisions
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

records = [
    json.loads(line)
    for line in trace_file.read_text().splitlines()
]

scheduled = [
    r for r in records
    if r["event"] == "request_scheduled"
]

print("Number of scheduling events:", len(scheduled))
print()

for r in scheduled[:50]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"state={r['source_state']:8s} | "
        f"new_tokens={r['num_new_tokens']:4d} | "
        f"computed={r['num_computed_tokens_before']:4d} | "
        f"prefill={str(r['is_prefill']):5s} | "
        f"chunk={str(r['is_prefill_chunk']):5s} | "
        f"budget_after={r['token_budget_after']:4d} | "
        f"free_blocks={r['free_kv_blocks']}"
    )

Number of scheduling events: 96

step= 1 | req=0-be824cc9 | state=WAITING  | new_tokens=2452 | computed=   0 | prefill=True  | chunk=False | budget_after=5740 | free_blocks=14187
step= 2 | req=0-be824cc9 | state=RUNNING  | new_tokens=   1 | computed=2452 | prefill=False | chunk=False | budget_after=8191 | free_blocks=14187
step= 2 | req=1-be780dff | state=WAITING  | new_tokens=   6 | computed=   0 | prefill=True  | chunk=False | budget_after=8185 | free_blocks=14186
step= 2 | req=2-90d9c3b4 | state=WAITING  | new_tokens=   5 | computed=   0 | prefill=True  | chunk=False | budget_after=8180 | free_blocks=14185
step= 3 | req=0-be824cc9 | state=RUNNING  | new_tokens=   1 | computed=2453 | prefill=False | chunk=False | budget_after=8191 | free_blocks=14185
step= 3 | req=1-be780dff | state=RUNNING  | new_tokens=   1 | computed=   6 | prefill=False | chunk=False | budget_after=8190 | free_blocks=14185
step= 3 | req=2-90d9c3b4 | state=RUNNING  | new_tokens=   1 | computed=   5 | prefill=False

### chunked prefill

In [54]:
# =============================================================================
# Inspect current scheduler configuration
# =============================================================================

%%writefile /content/inspect_scheduler_config.py

from vllm import LLM

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
)

config = llm.llm_engine.vllm_config.scheduler_config

print("max_num_batched_tokens:", config.max_num_batched_tokens)
print("max_num_scheduled_tokens:", config.max_num_scheduled_tokens)
print("enable_chunked_prefill:", config.enable_chunked_prefill)
print("long_prefill_token_threshold:", config.long_prefill_token_threshold)
print("max_num_seqs:", config.max_num_seqs)

Overwriting /content/inspect_scheduler_config.py


In [55]:
!python /content/inspect_scheduler_config.py

INFO 09-20 16:34:33 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:34:33 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:34:33 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:34:35 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:34:35 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:34:35 [model.py:2021] Using max model len 4096
INFO 09-20 16:34:35 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-20 16:34:36 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=15277) INFO 09-20 16:34:40 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model

In [58]:
# =============================================================================
# Inspect mixed-workload scheduling decisions
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

records = [
    json.loads(line)
    for line in trace_file.read_text().splitlines()
]

scheduled = [
    r for r in records
    if r["event"] == "request_scheduled"
]

print("Number of scheduling events:", len(scheduled))
print()

for r in scheduled[:50]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"state={r['source_state']:8s} | "
        f"new_tokens={r['num_new_tokens']:4d} | "
        f"computed={r['num_computed_tokens_before']:4d} | "
        f"prefill={str(r['is_prefill']):5s} | "
        f"chunk={str(r['is_prefill_chunk']):5s} | "
        f"budget_after={r['token_budget_after']:4d} | "
        f"free_blocks={r['free_kv_blocks']}"
    )

Number of scheduling events: 96

step= 1 | req=0-be824cc9 | state=WAITING  | new_tokens=2452 | computed=   0 | prefill=True  | chunk=False | budget_after=5740 | free_blocks=14187
step= 2 | req=0-be824cc9 | state=RUNNING  | new_tokens=   1 | computed=2452 | prefill=False | chunk=False | budget_after=8191 | free_blocks=14187
step= 2 | req=1-be780dff | state=WAITING  | new_tokens=   6 | computed=   0 | prefill=True  | chunk=False | budget_after=8185 | free_blocks=14186
step= 2 | req=2-90d9c3b4 | state=WAITING  | new_tokens=   5 | computed=   0 | prefill=True  | chunk=False | budget_after=8180 | free_blocks=14185
step= 3 | req=0-be824cc9 | state=RUNNING  | new_tokens=   1 | computed=2453 | prefill=False | chunk=False | budget_after=8191 | free_blocks=14185
step= 3 | req=1-be780dff | state=RUNNING  | new_tokens=   1 | computed=   6 | prefill=False | chunk=False | budget_after=8190 | free_blocks=14185
step= 3 | req=2-90d9c3b4 | state=RUNNING  | new_tokens=   1 | computed=   5 | prefill=False

In [59]:
%%writefile /content/test_mixed_workload.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
    max_num_batched_tokens=512,
    enable_chunked_prefill=True,
)

# Verify the actual scheduler configuration used by this run.
config = llm.llm_engine.vllm_config.scheduler_config

print("Scheduler configuration:")
print("  max_num_batched_tokens:", config.max_num_batched_tokens)
print("  max_num_scheduled_tokens:", config.max_num_scheduled_tokens)
print("  enable_chunked_prefill:", config.enable_chunked_prefill)
print("  long_prefill_token_threshold:", config.long_prefill_token_threshold)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=32,
)

long_prompt = "Explain GPU memory hierarchy in detail. " * 350

prompts = [
    long_prompt,
    "What is a GPU warp?",
    "What is KV cache?",
]

outputs = llm.generate(
    prompts,
    sampling_params,
)

for output in outputs:
    print("=" * 80)
    print("Request ID:", output.request_id)
    print("Prompt tokens:", len(output.prompt_token_ids))
    print("Output tokens:", len(output.outputs[0].token_ids))

Overwriting /content/test_mixed_workload.py


In [64]:
# =============================================================================
# Reset scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"
trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Trace file exists after reset:", trace_file.exists())

Trace file exists after reset: False


In [65]:
!python /content/test_mixed_workload.py

INFO 09-20 16:48:06 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 512, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:48:06 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:48:06 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:48:07 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:48:07 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:48:07 [model.py:2021] Using max model len 4096
INFO 09-20 16:48:07 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 09-20 16:48:08 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=19236) INFO 09-20 16:48:13 [core.py:

In [66]:
# =============================================================================
# Inspect mixed-workload scheduling decisions
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

records = [
    json.loads(line)
    for line in trace_file.read_text().splitlines()
]

scheduled = [
    r for r in records
    if r["event"] == "request_scheduled"
]

print("Number of scheduling events:", len(scheduled))
print()

for r in scheduled[:50]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"state={r['source_state']:8s} | "
        f"new_tokens={r['num_new_tokens']:4d} | "
        f"computed={r['num_computed_tokens_before']:4d} | "
        f"prefill={str(r['is_prefill']):5s} | "
        f"chunk={str(r['is_prefill_chunk']):5s} | "
        f"budget_after={r['token_budget_after']:4d} | "
        f"free_blocks={r['free_kv_blocks']}"
    )

Number of scheduling events: 100

step= 1 | req=0-8288c545 | state=WAITING  | new_tokens= 512 | computed=   0 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15258
step= 2 | req=0-8288c545 | state=RUNNING  | new_tokens= 512 | computed= 512 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15226
step= 3 | req=0-8288c545 | state=RUNNING  | new_tokens= 512 | computed=1024 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15194
step= 4 | req=0-8288c545 | state=RUNNING  | new_tokens= 512 | computed=1536 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15162
step= 5 | req=0-8288c545 | state=RUNNING  | new_tokens= 404 | computed=2048 | prefill=True  | chunk=True  | budget_after= 108 | free_blocks=15136
step= 5 | req=1-816308cc | state=WAITING  | new_tokens=   6 | computed=   0 | prefill=True  | chunk=False | budget_after= 102 | free_blocks=15135
step= 5 | req=2-9ad3df97 | state=WAITING  | new_tokens=   5 | computed=   0 | prefill=True

In [67]:
# =============================================================================
# Fix is_prefill_chunk semantics for RUNNING requests
# =============================================================================

source = scheduler_path.read_text()

old = """                is_prefill_chunk=request.is_prefill_chunk,
"""

new = """                is_prefill_chunk=(
                    request.num_computed_tokens + num_new_tokens
                    < request.num_prompt_tokens
                ),
"""

if old in source:
    source = source.replace(old, new, 1)
    scheduler_path.write_text(source)
    print("RUNNING is_prefill_chunk logic updated.")
else:
    print("Target line not found or already updated.")

RUNNING is_prefill_chunk logic updated.


In [68]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [72]:
# =============================================================================
# Reset scheduler trace
# =============================================================================

from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"
trace_file = Path(TRACE_PATH)

if trace_file.exists():
    trace_file.unlink()

print("Trace file exists after reset:", trace_file.exists())

Trace file exists after reset: False


In [73]:
!python /content/test_mixed_workload.py

INFO 09-20 16:54:05 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 512, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-20 16:54:05 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE
WARNING 09-20 16:54:05 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_SCHEDULER_TRACE_PATH
INFO 09-20 16:54:06 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-20 16:54:06 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-20 16:54:06 [model.py:2021] Using max model len 4096
INFO 09-20 16:54:06 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=512.
INFO 09-20 16:54:07 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=20889) INFO 09-20 16:54:12 [core.py:

In [74]:
# =============================================================================
# Inspect mixed-workload scheduling decisions
# =============================================================================

import json
from pathlib import Path

trace_file = Path("/content/scheduler_trace.jsonl")

records = [
    json.loads(line)
    for line in trace_file.read_text().splitlines()
]

scheduled = [
    r for r in records
    if r["event"] == "request_scheduled"
]

print("Number of scheduling events:", len(scheduled))
print()

for r in scheduled[:50]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"state={r['source_state']:8s} | "
        f"new_tokens={r['num_new_tokens']:4d} | "
        f"computed={r['num_computed_tokens_before']:4d} | "
        f"prefill={str(r['is_prefill']):5s} | "
        f"chunk={str(r['is_prefill_chunk']):5s} | "
        f"budget_after={r['token_budget_after']:4d} | "
        f"free_blocks={r['free_kv_blocks']}"
    )

Number of scheduling events: 100

step= 1 | req=0-9925e697 | state=WAITING  | new_tokens= 512 | computed=   0 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15258
step= 2 | req=0-9925e697 | state=RUNNING  | new_tokens= 512 | computed= 512 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15226
step= 3 | req=0-9925e697 | state=RUNNING  | new_tokens= 512 | computed=1024 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15194
step= 4 | req=0-9925e697 | state=RUNNING  | new_tokens= 512 | computed=1536 | prefill=True  | chunk=True  | budget_after=   0 | free_blocks=15162
step= 5 | req=0-9925e697 | state=RUNNING  | new_tokens= 404 | computed=2048 | prefill=True  | chunk=False | budget_after= 108 | free_blocks=15136
step= 5 | req=1-82393c93 | state=WAITING  | new_tokens=   6 | computed=   0 | prefill=True  | chunk=False | budget_after= 102 | free_blocks=15135
step= 5 | req=2-a777c7c1 | state=WAITING  | new_tokens=   5 | computed=   0 | prefill=True

### 4. Trace Analyzer

In [75]:
# =============================================================================
# Load scheduler trace
# =============================================================================

import json
from pathlib import Path

TRACE_PATH = "/content/scheduler_trace.jsonl"

trace_file = Path(TRACE_PATH)

records = [
    json.loads(line)
    for line in trace_file.read_text().splitlines()
]

print("Total trace records:", len(records))

Total trace records: 176


In [76]:
# =============================================================================
# Split trace by event type
# =============================================================================

schedule_start_events = [
    r for r in records
    if r["event"] == "schedule_start"
]

scheduled_events = [
    r for r in records
    if r["event"] == "request_scheduled"
]

preempted_events = [
    r for r in records
    if r["event"] == "request_preempted"
]

schedule_end_events = [
    r for r in records
    if r["event"] == "schedule_end"
]

print("Scheduler steps:", len(schedule_start_events))
print("Scheduling events:", len(scheduled_events))
print("Preemption events:", len(preempted_events))
print("Schedule-end events:", len(schedule_end_events))

Scheduler steps: 38
Scheduling events: 100
Preemption events: 0
Schedule-end events: 38


In [77]:
# =============================================================================
# Build per-request summary
# =============================================================================

from collections import defaultdict

request_stats = defaultdict(lambda: {
    "prompt_tokens": 0,
    "prefill_events": 0,
    "prefill_tokens": 0,
    "chunked_prefill_events": 0,
    "decode_events": 0,
    "decode_tokens": 0,
    "preemptions": 0,
})

for r in scheduled_events:
    req_id = r["request_id"]
    stats = request_stats[req_id]

    stats["prompt_tokens"] = r["num_prompt_tokens"]

    if r["is_prefill"]:
        stats["prefill_events"] += 1
        stats["prefill_tokens"] += r["num_new_tokens"]

        if r["is_prefill_chunk"]:
            stats["chunked_prefill_events"] += 1
    else:
        stats["decode_events"] += 1
        stats["decode_tokens"] += r["num_new_tokens"]

for r in preempted_events:
    request_stats[r["request_id"]]["preemptions"] += 1

for req_id, stats in request_stats.items():
    print("=" * 80)
    print("Request:", req_id)
    print("Prompt tokens:", stats["prompt_tokens"])
    print("Prefill events:", stats["prefill_events"])
    print("Prefill tokens scheduled:", stats["prefill_tokens"])
    print("Chunked prefill events:", stats["chunked_prefill_events"])
    print("Decode events:", stats["decode_events"])
    print("Decode tokens scheduled:", stats["decode_tokens"])
    print("Preemptions:", stats["preemptions"])

Request: 0-9925e697
Prompt tokens: 2452
Prefill events: 5
Prefill tokens scheduled: 2452
Chunked prefill events: 4
Decode events: 31
Decode tokens scheduled: 31
Preemptions: 0
Request: 1-82393c93
Prompt tokens: 6
Prefill events: 1
Prefill tokens scheduled: 6
Chunked prefill events: 0
Decode events: 31
Decode tokens scheduled: 31
Preemptions: 0
Request: 2-a777c7c1
Prompt tokens: 5
Prefill events: 1
Prefill tokens scheduled: 5
Chunked prefill events: 0
Decode events: 31
Decode tokens scheduled: 31
Preemptions: 0


In [78]:
# =============================================================================
# Scheduler-level summary
# =============================================================================

free_blocks = [
    r["free_kv_blocks"]
    for r in records
    if "free_kv_blocks" in r
]

total_scheduled_tokens = sum(
    r["num_new_tokens"]
    for r in scheduled_events
)

print("Total scheduler steps:", len(schedule_start_events))
print("Total requests:", len(request_stats))
print("Total scheduled tokens:", total_scheduled_tokens)
print("Total preemptions:", len(preempted_events))
print("Minimum free KV blocks:", min(free_blocks))
print("Maximum free KV blocks:", max(free_blocks))

Total scheduler steps: 38
Total requests: 3
Total scheduled tokens: 2556
Total preemptions: 0
Minimum free KV blocks: 15128
Maximum free KV blocks: 15290


In [79]:
# =============================================================================
# Print compact scheduling timeline
# =============================================================================

for r in scheduled_events:
    phase = "PREFILL" if r["is_prefill"] else "DECODE"

    if r["is_prefill"] and r["is_prefill_chunk"]:
        phase = "PREFILL_CHUNK"

    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"state={r['source_state']:8s} | "
        f"phase={phase:13s} | "
        f"tokens={r['num_new_tokens']:4d} | "
        f"computed={r['num_computed_tokens_before']:4d} | "
        f"budget_after={r['token_budget_after']:4d} | "
        f"free_blocks={r['free_kv_blocks']}"
    )

step= 1 | req=0-9925e697 | state=WAITING  | phase=PREFILL_CHUNK | tokens= 512 | computed=   0 | budget_after=   0 | free_blocks=15258
step= 2 | req=0-9925e697 | state=RUNNING  | phase=PREFILL_CHUNK | tokens= 512 | computed= 512 | budget_after=   0 | free_blocks=15226
step= 3 | req=0-9925e697 | state=RUNNING  | phase=PREFILL_CHUNK | tokens= 512 | computed=1024 | budget_after=   0 | free_blocks=15194
step= 4 | req=0-9925e697 | state=RUNNING  | phase=PREFILL_CHUNK | tokens= 512 | computed=1536 | budget_after=   0 | free_blocks=15162
step= 5 | req=0-9925e697 | state=RUNNING  | phase=PREFILL       | tokens= 404 | computed=2048 | budget_after= 108 | free_blocks=15136
step= 5 | req=1-82393c93 | state=WAITING  | phase=PREFILL       | tokens=   6 | computed=   0 | budget_after= 102 | free_blocks=15135
step= 5 | req=2-a777c7c1 | state=WAITING  | phase=PREFILL       | tokens=   5 | computed=   0 | budget_after=  97 | free_blocks=15134
step= 6 | req=0-9925e697 | state=RUNNING  | phase=DECODE      

In [80]:
# =============================================================================
# Consistency checks
# =============================================================================

from collections import defaultdict

# Group request_scheduled events by request.
events_by_request = defaultdict(list)

for r in scheduled_events:
    events_by_request[r["request_id"]].append(r)

errors = []

# -------------------------------------------------------------------------
# 1. Prefill tokens should match prompt tokens
# -------------------------------------------------------------------------
for req_id, events in events_by_request.items():
    prompt_tokens = events[0]["num_prompt_tokens"]

    prefill_tokens = sum(
        r["num_new_tokens"]
        for r in events
        if r["is_prefill"]
    )

    if prefill_tokens != prompt_tokens:
        errors.append(
            f"{req_id}: prefill_tokens={prefill_tokens}, "
            f"prompt_tokens={prompt_tokens}"
        )

# -------------------------------------------------------------------------
# 2. Per-step token budget accounting
# -------------------------------------------------------------------------
start_by_step = {
    r["step"]: r
    for r in schedule_start_events
}

end_by_step = {
    r["step"]: r
    for r in schedule_end_events
}

for step, start in start_by_step.items():
    if step not in end_by_step:
        errors.append(f"step {step}: missing schedule_end")
        continue

    end = end_by_step[step]

    consumed = (
        start["token_budget_before"]
        - end["token_budget_after"]
    )

    expected = end["total_num_scheduled_tokens"]

    if consumed != expected:
        errors.append(
            f"step {step}: budget consumed={consumed}, "
            f"scheduled={expected}"
        )

# -------------------------------------------------------------------------
# 3. Token budget should never become negative
# -------------------------------------------------------------------------
for r in records:
    if "token_budget_after" in r and r["token_budget_after"] < 0:
        errors.append(
            f"step {r['step']}: negative token budget"
        )

# -------------------------------------------------------------------------
# 4. num_computed_tokens should be monotonic per request
# -------------------------------------------------------------------------
for req_id, events in events_by_request.items():
    previous = -1

    for r in events:
        current = r["num_computed_tokens_before"]

        if current < previous:
            errors.append(
                f"{req_id}: computed tokens decreased "
                f"from {previous} to {current}"
            )

        previous = current

# -------------------------------------------------------------------------
# 5. Preemption count sanity check
# -------------------------------------------------------------------------
trace_preemptions = len(preempted_events)

schedule_end_preemptions = sum(
    r["preempted_count"]
    for r in schedule_end_events
)

if trace_preemptions != schedule_end_preemptions:
    errors.append(
        f"preemption mismatch: "
        f"trace={trace_preemptions}, "
        f"schedule_end={schedule_end_preemptions}"
    )

# -------------------------------------------------------------------------
# Report
# -------------------------------------------------------------------------

print("=" * 80)
print("Consistency Check")
print("=" * 80)

if not errors:
    print("PASS — all checks succeeded.")
else:
    print(f"FAIL — {len(errors)} issue(s) found:")
    for error in errors:
        print("-", error)

Consistency Check
PASS — all checks succeeded.
